# Lesson 1: simple Python notebook
Neeraj Bhuma

This notebook puts the work from the Python files in one place. Run from top to bottom in Databricks Free Edition Serverless. The two earlier notebooks are kept separately.

The column names follow Microsoft's AdventureWorks schema. SQL is used for joins and summaries. Prices use the same decimal types as the earlier version.

## 1. Set up the catalog and download the files

In [ ]:
import urllib.request
import matplotlib.pyplot as plt

spark.sql("CREATE CATALOG IF NOT EXISTS adventure_works")
spark.sql("USE CATALOG adventure_works")
spark.sql("CREATE SCHEMA IF NOT EXISTS adventure_works.default")
spark.sql("CREATE SCHEMA IF NOT EXISTS adventure_works.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS adventure_works.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS adventure_works.gold")
spark.sql("USE SCHEMA default")
spark.sql("CREATE VOLUME IF NOT EXISTS adventure_works.default.raw_data")

folder = "/Volumes/adventure_works/default/raw_data/"
source = "https://raw.githubusercontent.com/microsoft/sql-server-samples/master/samples/databases/adventure-works/oltp-install-script/"
files = ["SalesOrderDetail.csv", "SalesOrderHeader.csv", "Product.csv",
         "SalesTerritory.csv", "PurchaseOrderDetail.csv",
         "PurchaseOrderHeader.csv", "Vendor.csv", "instawdb.sql"]
for file in files:
    urllib.request.urlretrieve(source + file, folder + file)

print("Files downloaded")
display(dbutils.fs.ls(folder))

## 2. Read the sales files
Each file is tab-separated and has no header. The casts below give each column its name and type. This is longer than guessing column types, but the mapping is visible.

In [ ]:
# Read Product.
raw_data = spark.read.option("sep", "\t").option("header", False).csv(folder + "Product.csv")
raw_data.createOrReplaceTempView("raw_products")

In [ ]:
products = spark.sql("""
SELECT
    CAST(_c0 AS INT) AS `ProductID`,
    CAST(_c1 AS STRING) AS `Name`,
    CAST(_c2 AS STRING) AS `ProductNumber`,
    CAST(_c3 AS BOOLEAN) AS `MakeFlag`,
    CAST(_c4 AS BOOLEAN) AS `FinishedGoodsFlag`,
    CAST(_c5 AS STRING) AS `Color`,
    CAST(_c6 AS INT) AS `SafetyStockLevel`,
    CAST(_c7 AS INT) AS `ReorderPoint`,
    CAST(_c8 AS DECIMAL(19,4)) AS `StandardCost`,
    CAST(_c9 AS DECIMAL(19,4)) AS `ListPrice`,
    CAST(_c10 AS STRING) AS `Size`,
    CAST(_c11 AS STRING) AS `SizeUnitMeasureCode`,
    CAST(_c12 AS STRING) AS `WeightUnitMeasureCode`,
    CAST(_c13 AS DECIMAL(18,4)) AS `Weight`,
    CAST(_c14 AS INT) AS `DaysToManufacture`,
    CAST(_c15 AS STRING) AS `ProductLine`,
    CAST(_c16 AS STRING) AS `Class`,
    CAST(_c17 AS STRING) AS `Style`,
    CAST(_c18 AS INT) AS `ProductSubcategoryID`,
    CAST(_c19 AS INT) AS `ProductModelID`,
    CAST(_c20 AS TIMESTAMP) AS `SellStartDate`,
    CAST(_c21 AS TIMESTAMP) AS `SellEndDate`,
    CAST(_c22 AS TIMESTAMP) AS `DiscontinuedDate`,
    CAST(_c23 AS STRING) AS `rowguid`,
    CAST(_c24 AS TIMESTAMP) AS `ModifiedDate`
FROM raw_products
""")
products.createOrReplaceTempView("products")

In [ ]:
# Read SalesTerritory.
raw_data = spark.read.option("sep", "\t").option("header", False).csv(folder + "SalesTerritory.csv")
raw_data.createOrReplaceTempView("raw_territories")

In [ ]:
territories = spark.sql("""
SELECT
    CAST(_c0 AS INT) AS `TerritoryID`,
    CAST(_c1 AS STRING) AS `Name`,
    CAST(_c2 AS STRING) AS `CountryRegionCode`,
    CAST(_c3 AS STRING) AS `Group`,
    CAST(_c4 AS DECIMAL(19,4)) AS `SalesYTD`,
    CAST(_c5 AS DECIMAL(19,4)) AS `SalesLastYear`,
    CAST(_c6 AS DECIMAL(19,4)) AS `CostYTD`,
    CAST(_c7 AS DECIMAL(19,4)) AS `CostLastYear`,
    CAST(_c8 AS STRING) AS `rowguid`,
    CAST(_c9 AS TIMESTAMP) AS `ModifiedDate`
FROM raw_territories
""")
territories.createOrReplaceTempView("territories")

In [ ]:
# Read SalesOrderDetail.
raw_data = spark.read.option("sep", "\t").option("header", False).csv(folder + "SalesOrderDetail.csv")
raw_data.createOrReplaceTempView("raw_sales_detail")

In [ ]:
sales_detail = spark.sql("""
SELECT
    CAST(_c0 AS INT) AS `SalesOrderID`,
    CAST(_c1 AS INT) AS `SalesOrderDetailID`,
    CAST(_c2 AS STRING) AS `CarrierTrackingNumber`,
    CAST(_c3 AS INT) AS `OrderQty`,
    CAST(_c4 AS INT) AS `ProductID`,
    CAST(_c5 AS INT) AS `SpecialOfferID`,
    CAST(_c6 AS DECIMAL(19,4)) AS `UnitPrice`,
    CAST(_c7 AS DECIMAL(19,4)) AS `UnitPriceDiscount`,
    CAST(_c8 AS DECIMAL(38,6)) AS `LineTotal`,
    CAST(_c9 AS STRING) AS `rowguid`,
    CAST(_c10 AS TIMESTAMP) AS `ModifiedDate`
FROM raw_sales_detail
""")
sales_detail.createOrReplaceTempView("sales_detail")

In [ ]:
# Read SalesOrderHeader.
raw_data = spark.read.option("sep", "\t").option("header", False).csv(folder + "SalesOrderHeader.csv")
raw_data.createOrReplaceTempView("raw_sales_header")

In [ ]:
sales_header = spark.sql("""
SELECT
    CAST(_c0 AS INT) AS `SalesOrderID`,
    CAST(_c1 AS INT) AS `RevisionNumber`,
    CAST(_c2 AS TIMESTAMP) AS `OrderDate`,
    CAST(_c3 AS TIMESTAMP) AS `DueDate`,
    CAST(_c4 AS TIMESTAMP) AS `ShipDate`,
    CAST(_c5 AS INT) AS `Status`,
    CAST(_c6 AS BOOLEAN) AS `OnlineOrderFlag`,
    CAST(_c7 AS STRING) AS `SalesOrderNumber`,
    CAST(_c8 AS STRING) AS `PurchaseOrderNumber`,
    CAST(_c9 AS STRING) AS `AccountNumber`,
    CAST(_c10 AS INT) AS `CustomerID`,
    CAST(_c11 AS INT) AS `SalesPersonID`,
    CAST(_c12 AS INT) AS `TerritoryID`,
    CAST(_c13 AS INT) AS `BillToAddressID`,
    CAST(_c14 AS INT) AS `ShipToAddressID`,
    CAST(_c15 AS INT) AS `ShipMethodID`,
    CAST(_c16 AS INT) AS `CreditCardID`,
    CAST(_c17 AS STRING) AS `CreditCardApprovalCode`,
    CAST(_c18 AS INT) AS `CurrencyRateID`,
    CAST(_c19 AS DECIMAL(19,4)) AS `SubTotal`,
    CAST(_c20 AS DECIMAL(19,4)) AS `TaxAmt`,
    CAST(_c21 AS DECIMAL(19,4)) AS `Freight`,
    CAST(_c22 AS DECIMAL(19,4)) AS `TotalDue`,
    CAST(_c23 AS STRING) AS `Comment`,
    CAST(_c24 AS STRING) AS `rowguid`,
    CAST(_c25 AS TIMESTAMP) AS `ModifiedDate`
FROM raw_sales_header
""")
sales_header.createOrReplaceTempView("sales_header")

## 3. Build the sales star
One fact row is one sales-order line. The order header supplies OrderDate and TerritoryID.

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS headers, COUNT(DISTINCT SalesOrderID) AS distinct_headers FROM sales_header
"""))

In [ ]:
fact_sales = spark.sql("""
SELECT d.*, h.OrderDate, h.TerritoryID
FROM sales_detail d
JOIN sales_header h ON d.SalesOrderID = h.SalesOrderID
""")
fact_sales.createOrReplaceTempView("fact_sales")

In [ ]:
products.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.default.dim_product")

In [ ]:
territories.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.default.dim_sales_territory")

In [ ]:
fact_sales.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.default.fact_sales")

In [ ]:
print("Workspace stores folders, notebooks and Git folders.")
print("Catalog shows tables, schemas and volumes.")
print("Compute runs the notebook. This notebook uses Serverless.")
print("Jobs and Pipelines schedule work.")
print("Data Ingestion brings source files into the workspace.")

In [ ]:
display(spark.sql("""
SHOW TABLES IN adventure_works.default
"""))

In [ ]:
display(spark.sql("""
DESCRIBE adventure_works.default.fact_sales
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS row_count FROM adventure_works.default.fact_sales
"""))

In [ ]:
display(spark.sql("""
DESCRIBE adventure_works.default.dim_product
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS row_count FROM adventure_works.default.dim_product
"""))

In [ ]:
display(spark.sql("""
DESCRIBE adventure_works.default.dim_sales_territory
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS row_count FROM adventure_works.default.dim_sales_territory
"""))

## 4. Check keys and joins
A duplicate dimension key matches a sales line more than once. That repeats its revenue. An unsold product is different: it is in the product table but has no sales.

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT SalesOrderDetailID) AS distinct_detail_ids FROM fact_sales
"""))

In [ ]:
display(spark.sql("""
SELECT ProductID, COUNT(*) AS copies
FROM dim_product
GROUP BY ProductID
HAVING COUNT(*) > 1
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS left_join_rows
FROM fact_sales f
LEFT JOIN dim_product d ON f.ProductID = d.ProductID
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS right_join_rows
FROM fact_sales f
RIGHT JOIN dim_product d ON f.ProductID = d.ProductID
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS orphan_fact_rows
FROM fact_sales f
LEFT JOIN dim_product d ON f.ProductID = d.ProductID
WHERE d.ProductID IS NULL
"""))

In [ ]:
display(spark.sql("""
SELECT TerritoryID, COUNT(*) AS copies
FROM dim_sales_territory
GROUP BY TerritoryID
HAVING COUNT(*) > 1
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS left_join_rows
FROM fact_sales f
LEFT JOIN dim_sales_territory d ON f.TerritoryID = d.TerritoryID
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS right_join_rows
FROM fact_sales f
RIGHT JOIN dim_sales_territory d ON f.TerritoryID = d.TerritoryID
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS orphan_fact_rows
FROM fact_sales f
LEFT JOIN dim_sales_territory d ON f.TerritoryID = d.TerritoryID
WHERE d.TerritoryID IS NULL
"""))

## 5. First queries and revenue chart

In [ ]:
display(spark.sql("""
SELECT * FROM fact_sales
ORDER BY ModifiedDate DESC, SalesOrderDetailID DESC
LIMIT 10
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS sales_rows, COUNT(DISTINCT ProductID) AS products_sold FROM fact_sales
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(DISTINCT ProductID) AS catalog_products FROM dim_product
"""))

In [ ]:
revenue = spark.sql("""
SELECT f.ProductID, p.Name, SUM(f.LineTotal) AS Revenue
FROM fact_sales f
JOIN dim_product p ON f.ProductID = p.ProductID
GROUP BY f.ProductID, p.Name
ORDER BY Revenue DESC
""")
revenue.createOrReplaceTempView("revenue")
display(revenue.limit(10))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS bad_rows
FROM fact_sales
WHERE LineTotal IS NULL OR LineTotal < 0
"""))

In [ ]:
print("Check missing or negative amounts before adding revenue.")
print("Do not turn a missing amount into zero without a rule.")
top_five = revenue.limit(5).toPandas()
plt.figure(figsize=(10, 4))
plt.barh(top_five["Name"], top_five["Revenue"].astype(float))
plt.gca().invert_yaxis()
plt.xlabel("Net sales revenue (source monetary units)")
plt.title("Top 5 products by revenue")
plt.tight_layout()
plt.show()
plt.close()

## 6. Profile the sales star

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS orphaned_sales
FROM fact_sales f
LEFT JOIN dim_product p ON f.ProductID = p.ProductID
WHERE p.ProductID IS NULL
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS unsold_products
FROM dim_product p
LEFT JOIN fact_sales f ON p.ProductID = f.ProductID
WHERE f.ProductID IS NULL
"""))

In [ ]:
display(spark.sql("""
SELECT f.ProductID, p.Name, COUNT(*) AS order_line_count,
       COUNT(DISTINCT f.SalesOrderID) AS distinct_orders,
       SUM(f.OrderQty) AS units
FROM fact_sales f
JOIN dim_product p ON f.ProductID = p.ProductID
GROUP BY f.ProductID, p.Name
ORDER BY order_line_count DESC
LIMIT 10
"""))

In [ ]:
display(spark.sql("""
SELECT COALESCE(p.ProductLine, 'Unspecified') AS ProductLine,
       SUM(f.LineTotal) AS revenue
FROM fact_sales f
JOIN dim_product p ON f.ProductID = p.ProductID
GROUP BY p.ProductLine
ORDER BY revenue DESC
"""))

## 7. Purchasing homework
Bronze keeps the raw values. Silver adds names and types, then removes only NULL ProductID or negative LineTotal. A NULL LineTotal is kept, as in the original rule.

In [ ]:
# Read PurchaseOrderDetail.
raw_data = spark.read.option("sep", "\t").option("header", False).csv(folder + "PurchaseOrderDetail.csv")
raw_data.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.bronze.stg_purchase_order_detail")
raw_data.createOrReplaceTempView("raw_purchase_detail")

In [ ]:
purchase_detail = spark.sql("""
SELECT
    CAST(_c0 AS INT) AS `PurchaseOrderID`,
    CAST(_c1 AS INT) AS `PurchaseOrderDetailID`,
    CAST(_c2 AS TIMESTAMP) AS `DueDate`,
    CAST(_c3 AS INT) AS `OrderQty`,
    CAST(_c4 AS INT) AS `ProductID`,
    CAST(_c5 AS DECIMAL(19,4)) AS `UnitPrice`,
    CAST(_c6 AS DECIMAL(19,4)) AS `LineTotal`,
    CAST(_c7 AS DECIMAL(18,4)) AS `ReceivedQty`,
    CAST(_c8 AS DECIMAL(18,4)) AS `RejectedQty`,
    CAST(_c9 AS DECIMAL(18,4)) AS `StockedQty`,
    CAST(_c10 AS TIMESTAMP) AS `ModifiedDate`
FROM raw_purchase_detail
""")
purchase_detail.createOrReplaceTempView("purchase_detail")

In [ ]:
# Read PurchaseOrderHeader.
raw_data = spark.read.option("sep", "\t").option("header", False).csv(folder + "PurchaseOrderHeader.csv")
raw_data.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.bronze.stg_purchase_order_header")
raw_data.createOrReplaceTempView("raw_purchase_header")

In [ ]:
purchase_header = spark.sql("""
SELECT
    CAST(_c0 AS INT) AS `PurchaseOrderID`,
    CAST(_c1 AS INT) AS `RevisionNumber`,
    CAST(_c2 AS INT) AS `Status`,
    CAST(_c3 AS INT) AS `EmployeeID`,
    CAST(_c4 AS INT) AS `VendorID`,
    CAST(_c5 AS INT) AS `ShipMethodID`,
    CAST(_c6 AS TIMESTAMP) AS `OrderDate`,
    CAST(_c7 AS TIMESTAMP) AS `ShipDate`,
    CAST(_c8 AS DECIMAL(19,4)) AS `SubTotal`,
    CAST(_c9 AS DECIMAL(19,4)) AS `TaxAmt`,
    CAST(_c10 AS DECIMAL(19,4)) AS `Freight`,
    CAST(_c11 AS DECIMAL(19,4)) AS `TotalDue`,
    CAST(_c12 AS TIMESTAMP) AS `ModifiedDate`
FROM raw_purchase_header
""")
purchase_header.createOrReplaceTempView("purchase_header")

In [ ]:
# Read Vendor.
raw_data = spark.read.option("sep", "\t").option("header", False).csv(folder + "Vendor.csv")
raw_data.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.bronze.dim_vendor")
raw_data.createOrReplaceTempView("raw_vendors")

In [ ]:
vendors = spark.sql("""
SELECT
    CAST(_c0 AS INT) AS `BusinessEntityID`,
    CAST(_c1 AS STRING) AS `AccountNumber`,
    CAST(_c2 AS STRING) AS `Name`,
    CAST(_c3 AS INT) AS `CreditRating`,
    CAST(_c4 AS BOOLEAN) AS `PreferredVendorStatus`,
    CAST(_c5 AS BOOLEAN) AS `ActiveFlag`,
    CAST(_c6 AS STRING) AS `PurchasingWebServiceURL`,
    CAST(_c7 AS TIMESTAMP) AS `ModifiedDate`
FROM raw_vendors
""")
vendors.createOrReplaceTempView("vendors")

In [ ]:
print("Purchase lines before cleaning:", purchase_detail.count())
purchase_detail = purchase_detail.where("ProductID IS NOT NULL AND (LineTotal IS NULL OR LineTotal >= 0)")
print("Purchase lines after cleaning:", purchase_detail.count())
purchase_detail.createOrReplaceTempView("purchase_detail")

In [ ]:
purchase_detail.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.silver.stg_purchase_order_detail")

In [ ]:
purchase_header.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.silver.stg_purchase_order_header")

In [ ]:
vendors.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.silver.dim_vendor")

In [ ]:
products.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.silver.dim_product")

In [ ]:
territories.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.silver.dim_sales_territory")

In [ ]:
fact_purchases = spark.sql("""
SELECT d.*, h.VendorID, h.OrderDate
FROM purchase_detail d
JOIN purchase_header h ON d.PurchaseOrderID = h.PurchaseOrderID
""")
fact_purchases.createOrReplaceTempView("fact_purchases")

In [ ]:
fact_purchases.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.gold.fact_purchases")

In [ ]:
fact_sales.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.gold.fact_sales")

### Check the purchasing star

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS headers, COUNT(DISTINCT PurchaseOrderID) AS distinct_headers FROM purchase_header
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS purchase_lines FROM purchase_detail
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS purchase_fact_lines FROM fact_purchases
"""))

In [ ]:
display(spark.sql("""
SELECT PurchaseOrderID, PurchaseOrderDetailID, COUNT(*) AS copies
FROM fact_purchases
GROUP BY PurchaseOrderID, PurchaseOrderDetailID
HAVING COUNT(*) > 1
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS missing_products
FROM fact_purchases f
LEFT JOIN products p ON f.ProductID = p.ProductID
WHERE p.ProductID IS NULL
"""))

In [ ]:
display(spark.sql("""
SELECT COUNT(*) AS missing_vendors
FROM fact_purchases f
LEFT JOIN vendors v ON f.VendorID = v.BusinessEntityID
WHERE v.BusinessEntityID IS NULL
"""))

### Query 1: lowest margin per unit
Divide total net value by total quantity on each side. Summarize sales and purchases before joining, so rows do not multiply. Negative values are not a full accounting profit measure.

In [ ]:
sales_prices = spark.sql("""
SELECT ProductID, SUM(LineTotal) / SUM(OrderQty) AS sales_price
FROM fact_sales
GROUP BY ProductID
""")
sales_prices.createOrReplaceTempView("sales_prices")

purchase_prices = spark.sql("""
SELECT ProductID, SUM(LineTotal) / SUM(OrderQty) AS purchase_price
FROM fact_purchases
GROUP BY ProductID
""")
purchase_prices.createOrReplaceTempView("purchase_prices")

In [ ]:
lowest_margins = spark.sql("""
SELECT s.ProductID, d.Name, s.sales_price, p.purchase_price,
       s.sales_price - p.purchase_price AS margin_per_unit
FROM sales_prices s
JOIN purchase_prices p ON s.ProductID = p.ProductID
JOIN products d ON s.ProductID = d.ProductID
ORDER BY margin_per_unit, s.ProductID
LIMIT 5
""")
display(lowest_margins)

In [ ]:
lowest_margins.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.gold.query1_lowest_margins")

### Query 2: months with more sold than purchased
Use a full outer join to keep a month when only one side exists. Missing quantities become zero. Purchases mean ordered units, not received units. This gap alone does not prove a stockout.

In [ ]:
monthly_sales = spark.sql("""
SELECT ProductID, date_trunc('month', OrderDate) AS month,
       SUM(OrderQty) AS sold
FROM fact_sales
GROUP BY ProductID, date_trunc('month', OrderDate)
""")
monthly_sales.createOrReplaceTempView("monthly_sales")

monthly_purchases = spark.sql("""
SELECT ProductID, date_trunc('month', OrderDate) AS month,
       SUM(OrderQty) AS purchased
FROM fact_purchases
GROUP BY ProductID, date_trunc('month', OrderDate)
""")
monthly_purchases.createOrReplaceTempView("monthly_purchases")

In [ ]:
monthly_totals = spark.sql("""
SELECT COALESCE(s.ProductID, p.ProductID) AS ProductID,
       COALESCE(s.month, p.month) AS month,
       COALESCE(s.sold, 0) AS sold,
       COALESCE(p.purchased, 0) AS purchased
FROM monthly_sales s
FULL OUTER JOIN monthly_purchases p
ON s.ProductID = p.ProductID AND s.month = p.month
""")
monthly_totals.createOrReplaceTempView("monthly_totals")

In [ ]:
monthly_gaps = spark.sql("""
SELECT m.ProductID, m.month, m.sold, m.purchased, d.Name,
       m.sold - m.purchased AS quantity_gap
FROM monthly_totals m
JOIN products d ON m.ProductID = d.ProductID
WHERE m.sold > m.purchased
ORDER BY quantity_gap DESC, m.ProductID, m.month
""")
print("Product-months where sold is greater than purchased:", monthly_gaps.count())
display(monthly_gaps)

In [ ]:
monthly_gaps.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.gold.query2_monthly_quantity_gaps")

## 8. Team exercise

In [ ]:
print("Hello World")
print("Lakshmi Narayana Reddy Mandi")
print("Neeraj Bhuma")
print("Bala Sai Kiran")

## Sources
Microsoft AdventureWorks CSV files and schema:
https://github.com/microsoft/sql-server-samples/tree/master/samples/databases/adventure-works/oltp-install-script

This notebook needs Databricks, Spark and a catalog with volume permissions. It is not a standalone Python script. The original notebooks remain unchanged.